In [1]:
%cd ..

c:\Users\THIS PC\git\FMaFE


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os, gc
from pathlib import Path
from tqdm.auto import tqdm
from abc import ABC, abstractmethod

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA, PCA

# **Merge Data**


In [ ]:
X_PATH = [f'data/feat_matrix/raw/base/X/X{i}.pkl' for i in range(1, 12)]
Y_PATH = [f'data/feat_matrix/raw/base/Y/Y{i}.pkl' for i in range(1, 12)]
OUT_PATH = Path('data/feat_matrix/raw/reduced')
os.makedirs(OUT_PATH, exist_ok=True)

FEAT_DTYPE = np.float32
OUT_BIN = Path('feat_matrix.bin')

n_feats = None
patch_counts = []
labels_list = []

with open(OUT_PATH / OUT_BIN, 'wb') as out_f:
    n_images = 0
    for x_path, y_path in zip(X_PATH, Y_PATH):
        with open(x_path, 'rb') as f:
            X_shard = pickle.load(f)
        with open(y_path, 'rb') as f:
            Y_shard = pickle.load(f)

        for x, y in tqdm(zip(X_shard, Y_shard), total=len(X_shard),
                          desc=os.path.basename(x_path)):
            if n_feats is None:
                n_feats = x.shape[1]
            out_f.write(np.ascontiguousarray(x, dtype=FEAT_DTYPE).tobytes())
            patch_counts.append(x.shape[0])
            labels_list.append(np.asarray(y, dtype=np.int8))
            n_images += 1

        del X_shard, Y_shard
        gc.collect()

total_rows = sum(patch_counts)
print(f'{total_rows:,} rows x {n_feats} feats')

feat_mat = np.memmap(OUT_PATH / OUT_BIN, dtype=FEAT_DTYPE, mode='r', shape=(total_rows, n_feats))

image_id_arr = np.repeat(np.arange(n_images), patch_counts).astype(np.int32)
label_arr = np.concatenate(labels_list)

meta_df = pd.DataFrame({'image_id': image_id_arr, 'label': label_arr})
meta_df.to_parquet(OUT_PATH / 'meta.parquet')

In [3]:
OUT_PATH = Path('data/feat_matrix/raw/reduced')
os.makedirs(OUT_PATH, exist_ok=True)

print('\nLoading data...')
meta_df = pd.read_parquet(OUT_PATH / 'meta.parquet')
total_rows = len(meta_df)
n_feats = 637

feat_mat = np.memmap(OUT_PATH / 'feat_matrix.bin', dtype=np.float32, mode='r', shape=(total_rows, n_feats))


Loading data...


# **Data Preprocessing**


## **Data Splitter**

In [4]:
class SBSS:
    def __init__(
            self,
            feat_matrix: np.ndarray,
            meta_df: pd.DataFrame,
            pca_components: int = 50,
            random_state: int = 0,
            image_id: str = 'image_id',
            label_col: str = 'label'
    ):
        self.feat_matrix = feat_matrix
        self.meta_df = meta_df
        self.pca_components = pca_components
        self.random_state = random_state
        self.image_id = image_id
        self.label = label_col

    def image_fingerprint(self):
        scaler = StandardScaler()
        pca = IncrementalPCA(n_components=self.pca_components)
        
        batch_size = 100_000
        n_samples = self.feat_matrix.shape[0]
        
        print("Fitting Scaler incrementally...")
        for i in range(0, n_samples, batch_size):
            X_batch = self.feat_matrix[i:i+batch_size]
            scaler.partial_fit(X_batch)
            
        print("Fitting PCA incrementally...")
        for i in range(0, n_samples, batch_size):
            X_batch = self.feat_matrix[i:i+batch_size]
            X_scaled = scaler.transform(X_batch)
            pca.partial_fit(X_scaled)
            
        print("Extracting PCA features and aggregating per image...")
        image_ids = self.meta_df[self.image_id].values
        unique_images, inverse = np.unique(image_ids, return_inverse=True)
        num_images = len(unique_images)
        
        pca_sums = np.zeros((num_images, self.pca_components), dtype=np.float32)
        counts = np.bincount(inverse).astype(np.float32)
        
        for i in range(0, n_samples, batch_size):
            X_batch = self.feat_matrix[i:i+batch_size]
            X_scaled = scaler.transform(X_batch)
            X_pca = pca.transform(X_scaled)
            
            batch_inverse = inverse[i:i+batch_size]

            for j in range(self.pca_components):
                pca_sums[:, j] += np.bincount(batch_inverse, weights=X_pca[:, j], minlength=num_images)
                
        F_pca = pca_sums / counts[:, None]
        
        df_cluster = pd.DataFrame(F_pca, columns=[f'feat_{x}' for x in range(self.pca_components)])
        df_cluster[self.image_id] = unique_images
        
        C = self.meta_df.groupby(self.image_id).size().rename('Count')
        D = self.meta_df.groupby(self.image_id)[self.label].mean().rename('Density')
        
        df_stats = pd.concat([C, D], axis=1).reset_index()
        df_cluster = df_cluster.merge(df_stats, on=self.image_id)
        
        return df_cluster
    
    def kmeans(
            self,
            n_clusters: int = 5,
            plot: bool = True,
    ):
        df_cluster = self.image_fingerprint()
        X_cluster = df_cluster.drop(columns=[self.image_id]).to_numpy()
        
        km = KMeans(n_clusters=n_clusters, random_state=self.random_state)
        clusters = km.fit_predict(X_cluster)

        if plot:
            plot_pca = PCA(n_components=2)
            X_plot = plot_pca.fit_transform(X_cluster)
            plt.scatter(X_plot[:, 0], X_plot[:, 1], c=clusters)
            plt.xlabel('PCA 1')
            plt.ylabel('PCA 2')
            plt.title('K-Means Clustering', size=27)
            plt.tight_layout()
            plt.show()

        return clusters, df_cluster
    
    def _train_val_test_split(
            self,
            val_frac: float,
            test_frac: float,
    ):
        clusters, df_cluster = self.kmeans(plot=False)

        df_cluster['cluster_id'] = clusters
        df_cluster['split'] = 'train'
        
        for i in range(df_cluster['cluster_id'].nunique()):
            cluster_df = df_cluster[df_cluster['cluster_id'] == i]
            
            n_total = len(cluster_df)
            n_val = int(round(n_total * val_frac))
            n_test = int(round(n_total * test_frac))
            
            val_idx = cluster_df.sample(n=n_val, random_state=self.random_state).index
            df_cluster.loc[val_idx, 'split'] = 'val'
            
            remaining_df = cluster_df.drop(val_idx)
            test_idx = remaining_df.sample(n=n_test, random_state=self.random_state).index
            df_cluster.loc[test_idx, 'split'] = 'test'

        train_image_ids = df_cluster[self.image_id][df_cluster['split'] == 'train'].values
        val_image_ids = df_cluster[self.image_id][df_cluster['split'] == 'val'].values
        test_image_ids = df_cluster[self.image_id][df_cluster['split'] == 'test'].values

        train_mask = self.meta_df[self.image_id].isin(train_image_ids).values
        val_mask = self.meta_df[self.image_id].isin(val_image_ids).values
        test_mask = self.meta_df[self.image_id].isin(test_image_ids).values
        
        train_row_indices = np.where(train_mask)[0]
        val_row_indices = np.where(val_mask)[0]
        test_row_indices = np.where(test_mask)[0]
        
        y_train = self.meta_df[self.label].values[train_row_indices]
        y_val = self.meta_df[self.label].values[val_row_indices]
        y_test = self.meta_df[self.label].values[test_row_indices]

        del df_cluster, cluster_df, remaining_df 
        del train_mask, val_mask, test_mask
        gc.collect()

        return train_row_indices, val_row_indices, test_row_indices, y_train, y_val, y_test
    
    def train_val_test_split(
            self,
            val_frac: float = 0.1,
            test_frac: float = 0.1,
    ):
        assert 0.0 <= val_frac < 1.0, "val_frac must be in range [0, 1)."
        assert 0.0 <= test_frac < 1.0, "test_frac must be in range [0, 1)."
        assert val_frac + test_frac < 1.0, "val_frac + test_frac must be less than 1.0 to leave room for training data."

        return self._train_val_test_split(
            val_frac=val_frac,
            test_frac=test_frac
        )

In [5]:
print('\nSplitting data into train-val-test sets...')
sbss = SBSS(
    feat_mat,
    meta_df,
    pca_components=50,
    random_state=21,
    image_id='image_id',
    label_col='label'
)

train_idx, val_idx, test_idx, y_train, y_val, y_test = sbss.train_val_test_split(val_frac=0.1, test_frac=0.1)
print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}, Test size: {len(test_idx)}")

del sbss


Splitting data into train-val-test sets...
Fitting Scaler incrementally...
Fitting PCA incrementally...
Extracting PCA features and aggregating per image...
Train size: 8431774, Val size: 1073847, Test size: 1048735


## **Feature Selector**


In [6]:
class BaseFilterMethod(ABC):
    def __init__(self):
        self.score_dict = dict()

    @abstractmethod
    def score(
        self,
        X: np.ndarray,
        y: np.ndarray,
        indices: np.ndarray = None,
        bins: int = None
    ) -> dict[int, float]:
        ...

class OutOfCoreContingencyFilter(BaseFilterMethod):
    def _build_joint_counts(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray,
            bins: int,
            batch_size=100_000
        ):
        n_cols = X.shape[1]
        n_samples = len(indices) if indices is not None else X.shape[0]
        
        # 1. Map Y to discrete integer classes
        unique_y, y_mapped = np.unique(y, return_inverse=True)
        num_y = len(unique_y)
        
        # 2. Get global quantiles for X using a memory-safe random sample
        sample_size = min(1_000_000, n_samples)
        if indices is not None:
            sample_idx = np.random.choice(indices, size=sample_size, replace=False)
        else:
            sample_idx = np.random.choice(n_samples, size=sample_size, replace=False)
        sample_idx.sort()
        
        sample_buffer = []
        for i in range(0, sample_size, batch_size):
            sample_buffer.append(X[sample_idx[i:i+batch_size]])
        X_sample = np.vstack(sample_buffer)
        
        q = np.linspace(0, 100, bins + 1)
        quantiles_raw = np.percentile(X_sample, q, axis=0)
        
        # Handle columns with duplicate percentiles (e.g. constant columns)
        quantiles_list = []
        for c in range(n_cols):
            quantiles_list.append(np.unique(quantiles_raw[:, c]))
            
        # 3. Accumulate joint counts (X_binned, Y)
        num_x_bins = bins + 2 
        joint_counts = np.zeros((n_cols, num_x_bins, num_y), dtype=np.int64)
        loop_indices = indices if indices is not None else np.arange(n_samples)
        
        # Slice the array via rows
        for i in tqdm(range(0, n_samples, batch_size), desc='Reading Chunks', leave=False):
            batch_idx = loop_indices[i:i+batch_size]
            X_chunk = X[batch_idx]
            y_chunk_mapped = y_mapped[i:i+batch_size]
            
            for c in range(n_cols):
                # Bin the floats into discrete integers directly in RAM
                x_binned = np.digitize(X_chunk[:, c], quantiles_list[c])
                
                # Fast 2D frequency accumulation
                flat_indices = x_binned * num_y + y_chunk_mapped
                counts = np.bincount(flat_indices, minlength=num_x_bins * num_y)
                joint_counts[c] += counts.reshape(num_x_bins, num_y)
                
        return joint_counts

    def _compute_entropies(self, joint_counts):
        eps = 1e-9
        
        total = np.sum(joint_counts, axis=(1, 2), keepdims=True)
        P_xy = joint_counts / total
        
        P_x = np.sum(P_xy, axis=2) 
        P_y = np.sum(P_xy, axis=1) 
        
        H_x = -np.sum(P_x * np.log2(P_x + eps), axis=1)
        H_y = -np.sum(P_y * np.log2(P_y + eps), axis=1)
        H_xy = -np.sum(P_xy * np.log2(P_xy + eps), axis=(1, 2))
        
        return H_x, H_y, H_xy

class InformationGain(OutOfCoreContingencyFilter):
    def score(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None,
            bins: int = 10
        ):
        joint_counts = self._build_joint_counts(X, y, indices, bins)
        H_x, H_y, H_xy = self._compute_entropies(joint_counts)
        
        IG = H_x + H_y - H_xy
        
        for c in range(X.shape[1]):
            self.score_dict[c] = IG[c]
            
        return dict(sorted(self.score_dict.items(), key=lambda item: item[1], reverse=True))

class SymmetricUncertainty(OutOfCoreContingencyFilter):
    def score(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None,
            bins: int = 10
        ):
        joint_counts = self._build_joint_counts(X, y, indices, bins)
        H_x, H_y, H_xy = self._compute_entropies(joint_counts)
        
        IG = H_x + H_y - H_xy
        SU = 2 * IG / (H_x + H_y + 1e-9)
        
        for c in range(X.shape[1]):
            self.score_dict[c] = SU[c]
            
        return dict(sorted(self.score_dict.items(), key=lambda item: item[1], reverse=True))

class BaseScoreCombiner(ABC):
    @abstractmethod
    def combine(self, scores: list[dict[int, float]]) -> list[int]:
        ...

    def _cut(self, scores: dict[int, float]) -> list[int]:
        if getattr(self, 'top_k', None):
            return sorted(scores, key=scores.get, reverse=True)[:self.top_k]
        
        if getattr(self, 'threshold', None):
            return [k for k, v in scores.items() if v >= self.threshold]
        
        raise ValueError('Specify either top_k or threshold.')

class MeanCombiner(BaseScoreCombiner):
    def __init__(
            self,
            top_k: int = 1,
            threshold: float = None
        ):
        self.top_k = top_k
        self.threshold = threshold
    
    def combine(
            self,
            scores: list[dict[int, float]]
        ):
        keys = scores[0].keys()
        merged = {k: np.mean([s[k] for s in scores]) for k in keys}
        
        return self._cut(merged)
    
class IntersectCombiner(BaseScoreCombiner):
    def __init__(
            self,
            top_k: int = 1,
            min_agreement: int = 1
        ):
        self.top_k = top_k
        self.min_agreement = min_agreement

    def combine(
            self,
            scores: list[dict[int, float]]
        ):
        min_agree = self.min_agreement or len(scores)

        def top_keys(s):
            return set(sorted(s, key=s.get, reverse=True)[:self.top_k])
        
        agreement = {
            k: sum(k in top_keys(s) for s in scores)
            for k in scores[0].keys()
        }
        
        return [k for k, count in agreement.items() if count >= min_agree]
    
class Filter:
    def __init__(
            self,
            methods: list[BaseFilterMethod],
            combiner: BaseScoreCombiner = None
        ):
        self.methods = methods
        self.combiner = combiner
        self.selected_columns = None
    
    def fit(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None,
            n_bins: int = 10
        ):
        print('Start calculating feature scores...')
        
        scores = []
        for i, m in enumerate(self.methods):
            print(f"\n-> Running Filter Method {i+1}/{len(self.methods)}: {m.__class__.__name__}")
            scores.append(m.score(X, y, indices, bins=n_bins))
            
        print('\nChoosing columns to keep...')
        self.selected_columns = (
            list(scores[0].keys())
            if len(scores) == 1 and not self.combiner
            else self.combiner.combine(scores)
        )
        return self
    
    def transform(
            self,
            X: np.ndarray
        ):
        if self.selected_columns is None:
            raise RuntimeError("Call fit before transform")
        
        # NOTE: If X is a large memmap in size, this will return a new array in RAM.
        # Alternatively, just use the `filter.selected_columns` array!
        return X[:, self.selected_columns]
    
    def fit_transform(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None
        ):
        self.fit(X, y, indices)
        
        return self.transform(X)

In [7]:
top_k = 100
ig = InformationGain()
su = SymmetricUncertainty()
combiner = MeanCombiner(top_k=top_k)

print(f'\nFinding top {top_k} features...')
feature_filter = Filter(methods=[ig, su], combiner=combiner)
feature_filter.fit(X=feat_mat, y=y_train, indices=train_idx, n_bins=30)

selected_cols = feature_filter.selected_columns
print(f"Kept {len(selected_cols)} out of 637 columns.")


Finding top 100 features...
Start calculating feature scores...

-> Running Filter Method 1/2: InformationGain


Reading Chunks:   0%|          | 0/85 [00:00<?, ?it/s]


-> Running Filter Method 2/2: SymmetricUncertainty


Reading Chunks:   0%|          | 0/85 [00:00<?, ?it/s]


Choosing columns to keep...
Kept 100 out of 637 columns.


## **Data Balancer**

In [8]:
class BaseBalancer(ABC):
    @abstractmethod
    def fit_transform(self, *args, **kwargs) -> np.ndarray:
        ...

class ImageUndersamplingBinary(BaseBalancer):
    def fit_transform(
        self,
        indices: np.ndarray,
        meta_df: pd.DataFrame,
        ratio: float = 1.0,
        keep_original_ratio: float = 0.05,
        image_id_col: str = 'image_id',
        label_col: str = 'label',
        random_state: int = 42
    ) -> np.ndarray:
        
        print("Balancing dataset indices...")
        
        df = meta_df.iloc[indices].copy()
        
        kept_indices = []
        
        count_1 = df.groupby(image_id_col)[label_col].sum()
        count_0 = df.groupby(image_id_col)[label_col].count() - count_1
        
        manipulated_images = count_1[count_1 > 0].index
        original_images = count_1[count_1 == 0].index
        
        # ==========================================
        # A. Process Manipulated Images
        # ==========================================
        df_manipulated = df[df[image_id_col].isin(manipulated_images)]
        
        # Keep ALL label=1 patches
        df_1 = df_manipulated[df_manipulated[label_col] == 1]
        kept_indices.extend(df_1.index.values)
        
        # Undersample label=0 patches
        df_0_manipulated = df_manipulated[df_manipulated[label_col] == 0]
        
        target_0_counts = (count_1.loc[manipulated_images] * ratio).astype(int)
        target_0_counts = np.minimum(target_0_counts, count_0.loc[manipulated_images])
        
        if not df_0_manipulated.empty:
            df_0_man_shuffled = df_0_manipulated.sample(frac=1.0, random_state=random_state)
            df_0_man_shuffled['rank'] = df_0_man_shuffled.groupby(image_id_col).cumcount()
            
            allowed_counts = df_0_man_shuffled[image_id_col].map(target_0_counts)
            df_0_man_kept = df_0_man_shuffled[df_0_man_shuffled['rank'] < allowed_counts]
            
            kept_indices.extend(df_0_man_kept.index.values)
            
        # ==========================================
        # B. Process Original (Pristine) Images
        # ==========================================
        if keep_original_ratio > 0.0:
            df_original = df[df[image_id_col].isin(original_images)]
            
            if not df_original.empty:
                df_orig_shuffled = df_original.sample(frac=1.0, random_state=random_state)
                df_orig_shuffled['rank'] = df_orig_shuffled.groupby(image_id_col).cumcount()
                
                target_orig_counts = (count_0.loc[original_images] * keep_original_ratio).astype(int)
                allowed_orig_counts = df_orig_shuffled[image_id_col].map(target_orig_counts)
                
                df_orig_kept = df_orig_shuffled[df_orig_shuffled['rank'] < allowed_orig_counts]
                kept_indices.extend(df_orig_kept.index.values)

        balanced_indices = np.array(kept_indices, dtype=np.int64)
        np.random.seed(random_state)
        np.random.shuffle(balanced_indices)
        
        print(f"Original indices: {len(indices)} -> Balanced indices: {len(balanced_indices)}")
        return balanced_indices

In [9]:
print('\nUndersampling non-manipulated patches in training set...')
balancer = ImageUndersamplingBinary()
train_idx_bal = balancer.fit_transform(
    indices=train_idx, 
    meta_df=meta_df, 
    ratio=3.0,
    keep_original_ratio=0.05
)

y_train_bal = meta_df.loc[train_idx_bal, 'label'].values


Undersampling non-manipulated patches in training set...
Balancing dataset indices...
Original indices: 8431774 -> Balanced indices: 2417725


## **Data Normalizer**

In [10]:
class BaseNormalizer(ABC):
    @abstractmethod
    def fit(self, X: np.ndarray, indices: np.ndarray = None):
        ...
    
    @abstractmethod
    def partial_fit(self, X_batch: np.ndarray):
        ...
    
    @abstractmethod
    def transform(self, X: np.ndarray, out_path: str = None):
        ...

class RobustScaler(BaseNormalizer):
    def __init__(
            self,
            max_sample_size: int = 1_000_000,
            eps: float = 1e-8
        ):
        self.median = None
        self.IQR = None
        self.eps = eps
        
        self.max_sample_size = max_sample_size
        self._sample_buffer = []
        self._samples_seen = 0
    
    def partial_fit(
            self,
            X_batch: np.ndarray
        ):
        n_batch = X_batch.shape[0]
        
        if self._samples_seen < self.max_sample_size:
            take = min(self.max_sample_size - self._samples_seen, n_batch)
            idx = np.random.choice(n_batch, size=take, replace=False)
            self._sample_buffer.append(X_batch[idx].copy())
            self._samples_seen += take
            X_concat = np.vstack(self._sample_buffer)
            Q1 = np.percentile(X_concat, 25, axis=0)
            Q3 = np.percentile(X_concat, 75, axis=0)
            self.IQR = Q3 - Q1
            
            scale = np.median(np.abs(X_concat), axis=0)
            self.median = np.median(X_concat, axis=0)
            
            self.IQR = np.maximum(self.IQR, self.eps * scale)
            self.IQR[self.IQR == 0] = self.eps
            
        return self
    
    def fit(
            self,
            X: np.ndarray,
            indices: np.ndarray = None,
            batch_size: int = 100_000
        ):
        self._samples_seen = 0
        self._sample_buffer = []
        
        n_samples = len(indices) if indices is not None else X.shape[0]
        sample_size = min(self.max_sample_size, n_samples)
        
        # Calculate the percentage of rows needed to randomly sample from each batch
        fraction = sample_size / n_samples
        
        loop_indices = indices if indices is not None else np.arange(X.shape[0])
        
        # Sequential read to prevent disk thrashing
        for i in tqdm(range(0, n_samples, batch_size), desc="Fitting RobustScale", leave=False):
            batch_idx = loop_indices[i:i+batch_size]
            
            X_chunk = X[batch_idx]
            
            n_chunk = X_chunk.shape[0]
            take = int(n_chunk * fraction)
            if take > 0:
                local_idx = np.random.choice(n_chunk, size=take, replace=False)
                self._sample_buffer.append(X_chunk[local_idx].copy())
                self._samples_seen += take
                
        # Finalize medians
        X_concat = np.vstack(self._sample_buffer)
        Q1 = np.percentile(X_concat, 25, axis=0)
        Q3 = np.percentile(X_concat, 75, axis=0)
        self.IQR = Q3 - Q1
        
        scale = np.median(np.abs(X_concat), axis=0)
        self.median = np.median(X_concat, axis=0)
        
        self.IQR = np.maximum(self.IQR, self.eps * scale)
        self.IQR[self.IQR == 0] = self.eps
        
        return self
    
    def transform(
            self,
            X: np.ndarray,
            out_path: str = None,
            batch_size: int = 100_000
        ):
        if self.median is None or self.IQR is None:
            raise ValueError("RobustScale has not been fitted yet.")
            
        if out_path is not None:
            X_out = np.memmap(out_path, dtype=np.float32, mode='w+', shape=X.shape)
            n_samples = X.shape[0]
        
            for i in tqdm(range(0, n_samples, batch_size), desc="Writing scaled memmap", leave=False):
                X_out[i:i+batch_size] = (X[i:i+batch_size] - self.median) / self.IQR
            X_out.flush()
        
            return X_out
        
        else:
            return (X - self.median) / self.IQR

class StandardScaler(BaseNormalizer):
    def __init__(
            self,
            eps: float = 1e-8
        ):
        self.mean = None
        self.var = None
        self.eps = eps
        self.n_samples_seen = 0
    
    def partial_fit(self, X_batch: np.ndarray):
        if self.mean is None:
            self.mean = np.mean(X_batch, axis=0, dtype=np.float64)
            self.var = np.var(X_batch, axis=0, dtype=np.float64)
            self.n_samples_seen = X_batch.shape[0]
    
        else:
            n_a = self.n_samples_seen
            n_b = X_batch.shape[0]
            n_total = n_a + n_b
            
            mean_b = np.mean(X_batch, axis=0, dtype=np.float64)
            var_b = np.var(X_batch, axis=0, dtype=np.float64)
            
            delta = mean_b - self.mean
            self.mean = self.mean + delta * n_b / n_total
            
            m_a = self.var * n_a
            m_b = var_b * n_b
            M2 = m_a + m_b + (delta ** 2) * n_a * n_b / n_total
            self.var = M2 / n_total
            self.n_samples_seen = n_total
            
        return self
    
    def fit(
            self,
            X: np.ndarray,
            indices: np.ndarray = None,
            batch_size: int = 100_000
        ):
        self.mean = None
        self.var = None
        self.n_samples_seen = 0
        
        n_samples = len(indices) if indices is not None else X.shape[0]
        loop_indices = indices if indices is not None else np.arange(X.shape[0])
        
        for i in tqdm(range(0, n_samples, batch_size), desc="Fitting StandardScaler", leave=False):
            batch_idx = loop_indices[i:i+batch_size]
            self.partial_fit(X[batch_idx])
            
        return self
    
    @property
    def stdv(self):
        if self.var is None: return None
        std = np.sqrt(self.var)
        std = np.maximum(std, self.eps)
        std[std == 0] = self.eps
        return std
    
    def transform(
            self,
            X: np.ndarray,
            out_path: str = None,
            batch_size: int = 100_000
        ):
        if self.mean is None or self.var is None:
            raise ValueError("StandardScaler has not been fitted yet.")
            
        if out_path is not None:
            X_out = np.memmap(out_path, dtype=np.float32, mode='w+', shape=X.shape)
            n_samples = X.shape[0]
            for i in tqdm(range(0, n_samples, batch_size), desc="Writing scaled memmap", leave=False):
                X_out[i:i+batch_size] = (X[i:i+batch_size] - self.mean) / self.stdv
            X_out.flush()
        
            return X_out
        
        else: 
            return (X - self.mean) / self.stdv

In [11]:
print('\nScaling data...')
scaler = RobustScaler()
scaler.fit(feat_mat, indices=train_idx_bal)

scaler.median = scaler.median[selected_cols]
scaler.IQR = scaler.IQR[selected_cols]


Scaling data...


Fitting RobustScale:   0%|          | 0/25 [00:00<?, ?it/s]

## **Save Processed Data**

In [12]:
def export_split_to_disk(
    X_source: np.memmap, 
    meta_df: pd.DataFrame, 
    row_indices: np.ndarray, 
    selected_cols: list, 
    scaler, 
    out_x_path: str,
    out_y_path: str,
    batch_size: int = 100_000
):
    n_samples = len(row_indices)
    n_features = len(selected_cols)
    
    '''
    Exporting X to memmap data
    '''
    print(f"Exporting {n_samples} rows to {out_x_path}...")
    
    X_out = np.memmap(out_x_path, dtype=np.float32, mode='w+', shape=(n_samples, n_features))
    
    for i in tqdm(range(0, n_samples, batch_size), desc="Writing X chunks", leave=False):
        start = i
        end = min(i + batch_size, n_samples)
        batch_indices = row_indices[start:end]
        
        X_chunk = X_source[batch_indices][:, selected_cols]
        
        if scaler is not None:
            X_chunk = scaler.transform(X_chunk)
        
        X_out[start:end] = X_chunk
        
    X_out.flush()
    del X_out 
    
    '''
    Exporting y to parquet data
    '''
    print(f"Exporting metadata to {out_y_path}...")

    meta_split = meta_df.iloc[row_indices].copy()
    meta_split.to_parquet(out_y_path, index=False)
    
    print("Done!\n")

In [15]:
PROCESSED_PATH = Path('data/feat_matrix/processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH / 'train', exist_ok=True)
os.makedirs(PROCESSED_PATH / 'val', exist_ok=True)
os.makedirs(PROCESSED_PATH / 'test', exist_ok=True)

# 1. Save Training Set
export_split_to_disk(
    X_source=feat_mat,
    meta_df=meta_df,
    row_indices=train_idx_bal,
    selected_cols=selected_cols,
    scaler=scaler,
    out_x_path=(PROCESSED_PATH / 'train' / 'X_train.bin'),
    out_y_path=(PROCESSED_PATH / 'train' / 'y_train.parquet')
)
# 2. Save Validation Set
export_split_to_disk(
    X_source=feat_mat,
    meta_df=meta_df,
    row_indices=val_idx,
    selected_cols=selected_cols,
    scaler=scaler,
    out_x_path=(PROCESSED_PATH / 'val' / 'X_val.bin'),
    out_y_path=(PROCESSED_PATH / 'val' / 'y_val.parquet')
)
# 3. Save Test Set
export_split_to_disk(
    X_source=feat_mat,
    meta_df=meta_df,
    row_indices=test_idx,
    selected_cols=selected_cols,
    scaler=scaler,
    out_x_path=(PROCESSED_PATH / 'test' / 'X_test.bin'),
    out_y_path=(PROCESSED_PATH / 'test' / 'y_test.parquet')
)

Exporting 2417725 rows to data\feat_matrix\processed\train\X_train.bin...


Writing X chunks:   0%|          | 0/25 [00:00<?, ?it/s]

Exporting metadata to data\feat_matrix\processed\train\y_train.parquet...
Done!

Exporting 1073847 rows to data\feat_matrix\processed\val\X_val.bin...


Writing X chunks:   0%|          | 0/11 [00:00<?, ?it/s]

Exporting metadata to data\feat_matrix\processed\val\y_val.parquet...
Done!

Exporting 1048735 rows to data\feat_matrix\processed\test\X_test.bin...


Writing X chunks:   0%|          | 0/11 [00:00<?, ?it/s]

Exporting metadata to data\feat_matrix\processed\test\y_test.parquet...
Done!

